# Your Approach: Architecture Experiments on 40k Matched Data

**Task:** 3-class classification (human / ai / fusion).
**Fixed reference:** TF-IDF LR C=2.0, macro-F1 = **0.8549** on `val_fixed`.
**Encoder reference:** XLM-R head+tail 512, macro-F1 = **0.8971**.
**Data:** 40k train (`train_transformer_40k.csv`), 40,853 val (`val_fixed.csv`).

**Experiments:**
- **E1 — Chunked XLM-R:** split docs into 512-token chunks, encode each, pool with attention. Tests whether full-document coverage beats head+tail truncation.
- **E2 — Stylometry + XLM-R:** concatenate hand-crafted stylometric features (TTR, punctuation rates, sentence variance) with XLM-R [CLS] embedding.
- **E3 — Contrastive auxiliary loss:** add supervised contrastive loss to pull same-class embeddings together, targeting fusion/human confusion.
- **E4 — Longformer 4096:** sparse attention over full documents (if GPU memory permits).

**Artifacts saved to `/kaggle/working/experiments/`:**
- `e1_chunked/`, `e2_stylometry/`, `e3_contrastive/`, `e4_longformer/`
- `experiment_comparison.csv`
- `mistake_examples.json`

## 0. Shared Setup

Pinned to `transformers==4.57.1` (5.x breaks `TrainingArguments` on T4). Restart kernel after installing if needed.

In [1]:
# Run once, then restart kernel
# !pip install -q "transformers==4.57.1" "accelerate>=0.26.0"

import os, json, time, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LABELS = ["human", "ai", "fusion"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

INPUT = "/kaggle/input/datasets/suchok/base-train-data"
BASE  = "/kaggle/working"
EXP_DIR = f"{BASE}/experiments"
os.makedirs(EXP_DIR, exist_ok=True)

sub40 = pd.read_csv(f"{INPUT}/train_transformer_40k.csv")
val   = pd.read_csv(f"{INPUT}/val_fixed.csv")
sub40["label_id"] = sub40["label"].map(LABEL2ID)
val["label_id"]   = val["label"].map(LABEL2ID)

print(f"Train: {len(sub40)} | Val: {len(val)}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Label map: {LABEL2ID}")

Train: 40000 | Val: 40853
GPU: Tesla T4
Label map: {'human': 0, 'ai': 1, 'fusion': 2}


## E1 — Chunked XLM-R with Attention Pooling

**Motivation:** Head+tail 512 truncates 64% of documents and discards the middle. ChuLo shows that keyphrase-aware chunking outperforms truncation on long documents [citation:2][citation:9]. We split each document into overlapping 512-token chunks, encode each chunk with XLM-R, and learn an attention-weighted pool over chunk representations.

**Architecture:**
1. Tokenize full document without truncation.
2. Split `input_ids` into chunks of 512 with stride 256 (50% overlap).
3. Encode each chunk independently through XLM-R.
4. Attention-pool chunk [CLS] vectors → document representation.
5. Linear classifier on pooled vector.

**Cost:** ~3× forward passes per document (vs. single truncated pass).

In [2]:
MODEL_NAME = "xlm-roberta-base"
CHUNK_LEN  = 512
STRIDE     = 256
MAX_CHUNKS = 8   # cap to bound memory

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_document(text, max_chunks=MAX_CHUNKS):
    enc = tokenizer(text, truncation=False, add_special_tokens=False,
                    return_attention_mask=False)
    ids = enc["input_ids"]
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id

    if len(ids) <= CHUNK_LEN - 2:
        return [[cls_id] + ids + [sep_id]]

    chunks = []
    for start in range(0, len(ids), STRIDE):
        piece = ids[start:start + CHUNK_LEN - 2]
        if not piece:
            break
        chunks.append([cls_id] + piece + [sep_id])
        if len(chunks) >= max_chunks:
            break
    return chunks

class ChunkedDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return {"chunks": chunk_document(self.texts[idx]),
                "labels": int(self.labels[idx])}

def collate_chunks(batch):
    max_chunks = max(len(b["chunks"]) for b in batch)
    max_len = CHUNK_LEN
    bs = len(batch)
    input_ids = torch.full((bs, max_chunks, max_len), tokenizer.pad_token_id, dtype=torch.long)
    attn = torch.zeros((bs, max_chunks, max_len), dtype=torch.long)
    chunk_mask = torch.zeros((bs, max_chunks), dtype=torch.long)
    for i, b in enumerate(batch):
        for j, ch in enumerate(b["chunks"]):
            input_ids[i, j, :len(ch)] = torch.tensor(ch)
            attn[i, j, :len(ch)] = 1
            chunk_mask[i, j] = 1
    labels = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attn,
            "chunk_mask": chunk_mask, "labels": labels}

class ChunkedXLMR(nn.Module):
    def __init__(self, model_name, num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.attn_pool = nn.Linear(hidden, 1)
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask, chunk_mask, labels=None):
        bs, nc, sl = input_ids.shape
        flat_ids = input_ids.view(bs * nc, sl)
        flat_attn = attention_mask.view(bs * nc, sl)
        out = self.encoder(input_ids=flat_ids, attention_mask=flat_attn)
        cls = out.last_hidden_state[:, 0, :].view(bs, nc, -1)  # [B, NC, H]
        scores = self.attn_pool(cls).squeeze(-1)               # [B, NC]
        scores = scores.masked_fill(chunk_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        pooled = (cls * weights).sum(dim=1)                    # [B, H]
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            cw = torch.tensor([1.36, 0.745, 1.085], device=logits.device)
            loss = nn.CrossEntropyLoss(weight=cw)(logits, labels)
        return {"loss": loss, "logits": logits}

train_ds_e1 = ChunkedDataset(sub40["text"].tolist(), sub40["label_id"].tolist())
val_ds_e1   = ChunkedDataset(val["text"].tolist(),   val["label_id"].tolist())

# Sanity check one example
sample = train_ds_e1[0]
print(f"Example has {len(sample['chunks'])} chunks, first chunk len={len(sample['chunks'][0])}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2675 > 512). Running this sequence through the model will result in indexing errors


Example has 8 chunks, first chunk len=512


In [3]:
def compute_metrics_e1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    per = f1_score(labels, preds, average=None, labels=list(range(3)))
    return {"macro_f1": f1_score(labels, preds, average="macro"),
            "f1_human": per[0], "f1_ai": per[1], "f1_fusion": per[2]}

model_e1 = ChunkedXLMR(MODEL_NAME).to("cuda")

args_e1 = TrainingArguments(
    output_dir=f"{EXP_DIR}/e1_chunked",
    learning_rate=2e-5,
    per_device_train_batch_size=2,     # small: chunks × seq_len is memory-heavy
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,     # effective batch = 16
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.06,
    fp16=False,                        # safer for custom module
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer needs a thin wrapper to expose the model's forward signature
class ChunkedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

trainer_e1 = ChunkedTrainer(
    model=model_e1,
    args=args_e1,
    train_dataset=train_ds_e1,
    eval_dataset=val_ds_e1,
    data_collator=collate_chunks,
    compute_metrics=compute_metrics_e1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

print("E1 trainer ready. Training...")
trainer_e1.train()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E1 trainer ready. Training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,F1 Human,F1 Ai,F1 Fusion
1,0.219739,0.499594,0.837211,0.725978,0.974391,0.811265
2,0.163128,0.304310,0.902025,0.839321,0.989654,0.877100


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=2500, training_loss=0.2542177318572998, metrics={'train_runtime': 33111.467, 'train_samples_per_second': 2.416, 'train_steps_per_second': 0.076, 'total_flos': 0.0, 'train_loss': 0.2542177318572998, 'epoch': 2.0})

In [4]:
# E1 evaluation
pred_e1 = trainer_e1.predict(val_ds_e1)
y_pred_e1 = np.argmax(pred_e1.predictions, axis=-1)
y_true_e1 = val["label_id"].values

macro_e1 = f1_score(y_true_e1, y_pred_e1, average="macro")
per_e1 = f1_score(y_true_e1, y_pred_e1, average=None, labels=list(range(3)))
print(f"E1 Chunked: macro-F1={macro_e1:.4f} | "
      f"h={per_e1[0]:.3f} ai={per_e1[1]:.3f} f={per_e1[2]:.3f}")

# Save predictions for mistake analysis
np.save(f"{EXP_DIR}/e1_chunked/preds.npy", y_pred_e1)
trainer_e1.save_model(f"{EXP_DIR}/e1_chunked/final")
tokenizer.save_pretrained(f"{EXP_DIR}/e1_chunked/final")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


E1 Chunked: macro-F1=0.9020 | h=0.839 ai=0.990 f=0.877


('/kaggle/working/experiments/e1_chunked/final/tokenizer_config.json',
 '/kaggle/working/experiments/e1_chunked/final/tokenizer.json')

In [5]:
# Save the full custom model state dict and config
torch.save(model_e1.state_dict(), f"{EXP_DIR}/e1_chunked/model_state.pt")
tokenizer.save_pretrained(f"{EXP_DIR}/e1_chunked")
with open(f"{EXP_DIR}/e1_chunked/arch_config.json", "w") as f:
    json.dump({"chunk_len": CHUNK_LEN, "stride": STRIDE, "max_chunks": MAX_CHUNKS}, f)
print("E1 weights + tokenizer saved.")

E1 weights + tokenizer saved.


## E2 — Stylometry + XLM-R Hybrid

**Motivation:** Your baseline ablation showed **char-only (0.8395) > word-only (0.8286)** — punctuation and character patterns are discriminative. The hybrid framework paper confirms that combining stylometric features with transformer embeddings improves robustness [citation:5][citation:7].

**Features (18 total, language-agnostic):**
- **Lexical:** TTR, hapax ratio, mean word length, std word length
- **Punctuation rates:** em-dash, semicolon, ellipsis, question, exclaim, comma, colon, parenthesis (per 1k words)
- **Sentence-level:** mean sentences per doc, std sentence length, intra-doc sentence TTR variance
- **Structural:** newline count, digit ratio, uppercase ratio

**Architecture:** XLM-R [CLS] (768-dim) concatenated with 18 normalized stylometric features → 786-dim → linear classifier. XLM-R is fine-tuned end-to-end.

In [6]:
def extract_stylometric(text):
    words = text.split()
    nw = max(len(words), 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 3]
    ns = max(len(sents), 1)

    ttr = len(set(words)) / nw
    hapax = sum(1 for w in set(words) if words.count(w) == 1) / nw
    wlens = [len(w) for w in words] or [0]
    mean_wl, std_wl = float(np.mean(wlens)), float(np.std(wlens))

    def rate(ch): return text.count(ch) / nw * 1000
    punct = [rate("—"), rate(";"), rate("..."), rate("?"), rate("!"),
             rate(","), rate(":"), rate("(")]

    sent_lens = [len(s.split()) for s in sents] or [0]
    sent_ttrs = [len(s.split()) / max(len(set(s.split())), 1) for s in sents] or [0]

    feats = [
        ttr, hapax, mean_wl, std_wl,
        *punct,
        len(sents) / 10.0,
        float(np.mean(sent_lens)), float(np.std(sent_lens)),
        float(np.std(sent_ttrs)) if len(sent_ttrs) > 1 else 0.0,
        text.count("\n") / 10.0,
        sum(c.isdigit() for c in text) / max(len(text), 1),
        sum(c.isupper() for c in text) / max(len(text), 1),
    ]
    return feats

# Compute features for train and val
X_sty_train = np.array([extract_stylometric(t) for t in sub40["text"]])
X_sty_val   = np.array([extract_stylometric(t) for t in val["text"]])

# Normalize using train statistics only
mu, sigma = X_sty_train.mean(0), X_sty_train.std(0) + 1e-8
X_sty_train = (X_sty_train - mu) / sigma
X_sty_val   = (X_sty_val - mu) / sigma

print(f"Stylometric features: {X_sty_train.shape[1]} dims")
print(f"Sample feature vector (first 5): {X_sty_train[0, :5].round(3)}")

Stylometric features: 19 dims
Sample feature vector (first 5): [-1.589 -1.323 -0.655 -0.275 -0.175]


In [7]:
class HybridDataset(Dataset):
    def __init__(self, texts, sty_feats, labels, tokenizer, max_len=512):
        self.texts, self.sty, self.labels = texts, sty_feats, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True,
                             max_length=self.max_len, padding=False)
        enc["sty"] = torch.tensor(self.sty[idx], dtype=torch.float)
        enc["labels"] = int(self.labels[idx])
        return enc

class HybridStyloXLMR(nn.Module):
    def __init__(self, model_name, sty_dim, num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden + sty_dim, num_labels)
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask, sty, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        combined = torch.cat([cls, sty], dim=-1)
        logits = self.classifier(self.dropout(combined))
        loss = None
        if labels is not None:
            cw = torch.tensor([1.36, 0.745, 1.085], device=logits.device)
            loss = nn.CrossEntropyLoss(weight=cw)(logits, labels)
        return {"loss": loss, "logits": logits}

train_ds_e2 = HybridDataset(sub40["text"].tolist(), X_sty_train, sub40["label_id"].tolist(), tokenizer)
val_ds_e2   = HybridDataset(val["text"].tolist(),   X_sty_val,   val["label_id"].tolist(),   tokenizer)

model_e2 = HybridStyloXLMR(MODEL_NAME, sty_dim=X_sty_train.shape[1]).to("cuda")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
def compute_metrics_e2(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    per = f1_score(labels, preds, average=None, labels=list(range(3)))
    return {"macro_f1": f1_score(labels, preds, average="macro"),
            "f1_human": per[0], "f1_ai": per[1], "f1_fusion": per[2]}

args_e2 = TrainingArguments(
    output_dir=f"{EXP_DIR}/e2_stylometry",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.06,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

class HybridTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

trainer_e2 = HybridTrainer(
    model=model_e2, args=args_e2,
    train_dataset=train_ds_e2, eval_dataset=val_ds_e2,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics_e2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)
trainer_e2.train()

ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`sty` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:
# E2 evaluation
pred_e2 = trainer_e2.predict(val_ds_e2)
y_pred_e2 = np.argmax(pred_e2.predictions, axis=-1)
y_true_e2 = val["label_id"].values

macro_e2 = f1_score(y_true_e2, y_pred_e2, average="macro")
per_e2 = f1_score(y_true_e2, y_pred_e2, average=None, labels=list(range(3)))
print(f"E2 Stylometry: macro-F1={macro_e2:.4f} | "
      f"h={per_e2[0]:.3f} ai={per_e2[1]:.3f} f={per_e2[2]:.3f}")

np.save(f"{EXP_DIR}/e2_stylometry/preds.npy", y_pred_e2)
trainer_e2.save_model(f"{EXP_DIR}/e2_stylometry/final")

In [ ]:
# Save the full custom model state dict
torch.save(model_e2.state_dict(), f"{EXP_DIR}/e2_stylometry/model_state.pt")
tokenizer.save_pretrained(f"{EXP_DIR}/e2_stylometry")
# Save the stylometric normalization stats so inference matches training
np.savez(f"{EXP_DIR}/e2_stylometry/sty_stats.npz", mu=mu, sigma=sigma)
print("E2 weights, tokenizer, and stylometric stats saved.")

## E3 — Supervised Contrastive Auxiliary Loss

**Motivation:** FAID shows that contrastive learning improves fine-grained three-class detection, especially for human-LLM collaborative text [citation:3]. Your encoder's fusion F1 (0.865) lags human (0.837) and ai (0.989) in absolute improvement; contrastive loss directly targets class separation in embedding space.

**Loss:** `total = CE(logits, labels) + λ * SupCon(embeddings, labels)`
- `SupCon` pulls same-class [CLS] embeddings together and pushes different-class apart.
- λ = 0.1 (auxiliary, not dominant).
- Temperature τ = 0.07.

In [ ]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature
    def forward(self, features, labels):
        features = nn.functional.normalize(features, dim=1)
        sim = features @ features.T / self.tau
        mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        mask.fill_diagonal_(False)
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        pos_counts = mask.sum(1).clamp(min=1)
        loss = -(log_prob * mask).sum(1) / pos_counts
        return loss.mean()

class ContrastiveXLMR(nn.Module):
    def __init__(self, model_name, num_labels=3, lam=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)
        self.supcon = SupConLoss()
        self.lam = lam
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        loss = None
        if labels is not None:
            cw = torch.tensor([1.36, 0.745, 1.085], device=logits.device)
            ce = nn.CrossEntropyLoss(weight=cw)(logits, labels)
            con = self.supcon(cls, labels)
            loss = ce + self.lam * con
        return {"loss": loss, "logits": logits}

# Reuse E2 datasets (tokenization only, no stylometry)
train_ds_e3 = HybridDataset(sub40["text"].tolist(), np.zeros((len(sub40), 1)), sub40["label_id"].tolist(), tokenizer)
val_ds_e3   = HybridDataset(val["text"].tolist(),   np.zeros((len(val), 1)),   val["label_id"].tolist(),   tokenizer)

model_e3 = ContrastiveXLMR(MODEL_NAME).to("cuda")

args_e3 = TrainingArguments(
    output_dir=f"{EXP_DIR}/e3_contrastive",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.06,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

class ConTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

trainer_e3 = ConTrainer(
    model=model_e3, args=args_e3,
    train_dataset=train_ds_e3, eval_dataset=val_ds_e3,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics_e2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)
trainer_e3.train()

In [ ]:
pred_e3 = trainer_e3.predict(val_ds_e3)
y_pred_e3 = np.argmax(pred_e3.predictions, axis=-1)
y_true_e3 = val["label_id"].values
macro_e3 = f1_score(y_true_e3, y_pred_e3, average="macro")
per_e3 = f1_score(y_true_e3, y_pred_e3, average=None, labels=list(range(3)))
print(f"E3 Contrastive: macro-F1={macro_e3:.4f} | "
      f"h={per_e3[0]:.3f} ai={per_e3[1]:.3f} f={per_e3[2]:.3f}")
np.save(f"{EXP_DIR}/e3_contrastive/preds.npy", y_pred_e3)

In [ ]:
# E3 currently doesn't save anything — add this
torch.save(model_e3.state_dict(), f"{EXP_DIR}/e3_contrastive/model_state.pt")
tokenizer.save_pretrained(f"{EXP_DIR}/e3_contrastive")
print("E3 weights + tokenizer saved.")

## Comparison Table

All models trained on the **same 40k rows** and evaluated on the **same `val_fixed`**. The fixed reference is TF-IDF LR C=2.0 (0.8549). The encoder reference is XLM-R head+tail 512 (0.8971).

In [ ]:
comparison = pd.DataFrame([
    {"model": "Baseline TF-IDF (40k)", "macro_f1": 0.8549, "f1_human": 0.7882, "f1_ai": 0.9810, "f1_fusion": 0.7956},
    {"model": "Encoder XLM-R head+tail 512", "macro_f1": 0.8971, "f1_human": 0.8372, "f1_ai": 0.9891, "f1_fusion": 0.8650},
    {"model": "E1 Chunked XLM-R", "macro_f1": round(macro_e1, 4), "f1_human": round(per_e1[0], 4), "f1_ai": round(per_e1[1], 4), "f1_fusion": round(per_e1[2], 4)},
    {"model": "E2 Stylometry + XLM-R", "macro_f1": round(macro_e2, 4), "f1_human": round(per_e2[0], 4), "f1_ai": round(per_e2[1], 4), "f1_fusion": round(per_e2[2], 4)},
    {"model": "E3 Contrastive XLM-R", "macro_f1": round(macro_e3, 4), "f1_human": round(per_e3[0], 4), "f1_ai": round(per_e3[1], 4), "f1_fusion": round(per_e3[2], 4)},
]).sort_values("macro_f1", ascending=False)

print(comparison.to_string(index=False))
comparison.to_csv(f"{EXP_DIR}/experiment_comparison.csv", index=False)
print(f"\nSaved -> {EXP_DIR}/experiment_comparison.csv")

## Mistake Analysis: Where Models Disagree

For each experiment model, we identify validation examples where **that model is wrong but the encoder reference is correct** (or vice versa). We show **maximum 2 examples per model**, focusing on fusion and human confusions, which are the hardest classes per your EDA.

In [ ]:
# Reference: encoder XLM-R head+tail 512
ref_preds = np.load(f"{INPUT}/encoder_xlmr_40k_preds.npy") if os.path.exists(f"{INPUT}/encoder_xlmr_40k_preds.npy") else None
# If not available, use baseline TF-IDF predictions as fallback
if ref_preds is None:
    # Recompute from saved model if needed; for now, use a placeholder
    # In practice, load the encoder predictions you saved earlier
    print("Encoder predictions not found — using E2 as reference instead")
    ref_preds = y_pred_e2

val_labels = val["label_id"].values

def find_mistakes(model_preds, ref_preds, val_df, labels, model_name, max_examples=2):
    """Find examples where model is wrong but ref is right, or vice versa."""
    model_wrong = model_preds != val_labels
    ref_right = ref_preds == val_labels
    model_right = model_preds == val_labels
    ref_wrong = ref_preds != val_labels

    # Priority 1: model wrong, ref right (model's unique mistakes)
    unique_mistakes = np.where(model_wrong & ref_right)[0]
    # Priority 2: model right, ref wrong (model's unique successes)
    unique_success = np.where(model_right & ref_wrong)[0]

    examples = []
    for idx in list(unique_mistakes)[:max_examples]:
        examples.append({
            "idx": int(idx), "id": val_df.iloc[idx]["id"],
            "true": labels[val_labels[idx]],
            "model_pred": labels[model_preds[idx]],
            "ref_pred": labels[ref_preds[idx]],
            "domain": val_df.iloc[idx]["domain"],
            "lang": val_df.iloc[idx]["lang"],
            "text_snippet": val_df.iloc[idx]["text"][:400].replace("\n", " "),
            "type": f"{model_name} wrong, encoder right",
        })
    for idx in list(unique_success)[:max_examples - len(examples)]:
        examples.append({
            "idx": int(idx), "id": val_df.iloc[idx]["id"],
            "true": labels[val_labels[idx]],
            "model_pred": labels[model_preds[idx]],
            "ref_pred": labels[ref_preds[idx]],
            "domain": val_df.iloc[idx]["domain"],
            "lang": val_df.iloc[idx]["lang"],
            "text_snippet": val_df.iloc[idx]["text"][:400].replace("\n", " "),
            "type": f"{model_name} right, encoder wrong",
        })
    return examples

all_mistakes = {}
for name, preds in [("E1_Chunked", y_pred_e1), ("E2_Stylometry", y_pred_e2), ("E3_Contrastive", y_pred_e3)]:
    exs = find_mistakes(preds, ref_preds, val, LABELS, name, max_examples=2)
    all_mistakes[name] = exs
    print(f"\n=== {name} ({len(exs)} examples) ===")
    for e in exs:
        print(f"  [{e['type']}] true={e['true']} model={e['model_pred']} ref={e['ref_pred']}")
        print(f"    domain={e['domain']} lang={e['lang']}")
        print(f"    text: {e['text_snippet'][:200]}...")

with open(f"{EXP_DIR}/mistake_examples.json", "w") as f:
    json.dump(all_mistakes, f, indent=2, ensure_ascii=False)
print(f"\nSaved -> {EXP_DIR}/mistake_examples.json")

## Conclusions (fill in after running)

- **Best experiment:** `___` with macro-F1 = `___`, vs. encoder reference 0.8971.
- **Fusion improvement:** `___` changed fusion F1 from 0.8650 to `___`.
- **Long-text effect:** E1 chunked `___` vs. head+tail 512 — did full-document coverage help, and for which domains?
- **Stylometry effect:** E2 `___` — do hand-crafted features add signal on top of XLM-R, or does the transformer already capture them?
- **Contrastive effect:** E3 `___` — did the auxiliary loss reduce human/fusion confusion in the confusion matrix?
- **Error analysis:** the most common remaining confusion is `___` → `___`, concentrated in domain `___`.